# dim_edition.py

## Purpose
Builds `dim_edition` as an SCD1 dimension with all EA FC editions from Bronze.

## Source
- Bronze: `fifa.bronze.ea_players_raw` (column `edition`)

## Strategy
- SCD1 (editions are stable)
- Overwrite mode (more efficient for small catalogs)
- Surrogate key: year extracted from the edition code (`fc27` → `2027`)
- One row per edition, without `snapshot_date`

## Output
- Table: `fifa.silver.dim_edition`

## Columns
- `edition_id` (INT): year, e.g. 2027
- `edition_code` (STRING): e.g. "fc27"
- `edition_name` (STRING): e.g. "EA SPORTS FC 27"
- `_ingestion_timestamp` (TIMESTAMP)

## Why overwrite and not MERGE?

`dim_edition` is a tiny catalog. `overwrite` is cheaper and simpler than `MERGE` for tables this small. For larger catalogs (like `dim_nationality` with 164 rows), `MERGE` is preferred.


## Notes

### Year extraction limitation

The `edition_id` is derived by extracting the 2-digit year from the edition code and adding 2000:

```python
regexp_extract(col("edition"), r"fc(\d+)", 1).cast("int") + 2000
```

This project is designed for editions from 2000 onwards. Examples:

fc24 → 2024

fc27 → 2027

fc300 → 2300 (works, as long as the year is 2000+)
Editions beyond 2099 (e.g. fc00) would collide with 2000.

Not handled:

Pre-2000 editions (e.g. fifa99 → would be 2099 instead of 1999).

Why it's acceptable: the dataset and the project scope only cover modern FC editions. If historical editions are added in the future, the logic should be replaced with a full-year mapping or a lookup table.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

dim_edition = (
    bronze
    .select("edition")
    .filter(col("edition").isNotNull() & (col("edition") != ""))
    .distinct()
    # regexp_extract extracts the year from the edition column (e.g. "fc27" -> "27")
    .withColumn("edition_id",
        regexp_extract(col("edition"), r"fc(\d+)", 1).try_cast("int") + 2000
    )
    .withColumn("edition_code", col("edition"))
    .withColumn("edition_name",
        concat(lit("EA SPORTS FC "), regexp_extract(col("edition"), r"fc(\d+)", 1))
    )
    .withColumn("_ingestion_timestamp", current_timestamp())
    .select("edition_id", "edition_code", "edition_name", "_ingestion_timestamp")
    .orderBy("edition_id")
)


# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_edition (
    edition_id INT,
    edition_code STRING,
    edition_name STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# Overwrite the table (small catalog, more efficient than MERGE)
(dim_edition
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable("fifa.silver.dim_edition")
)

print(f"dim_edition creada con {spark.read.table('fifa.silver.dim_edition').count()} filas")


LIMIT 10 of the dim_edition table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_edition ORDER BY edition_id limit 10;